# First SNTP Event Display

This notebook loads the local ROOT SNTP file, finds the SNTP tree that contains the event-display branches, and plots the first event in the two MINOS perspectives.

The two panels share the plane axis on x, while the strip axis is shown on y with the second perspective mirrored to reflect the alternating detector orientation.
The full detector canvas is fixed to the MINOS Far Detector grid, with a visible blank band between planes 243 and 244 for the 1.5 m supermodule gap. Pulse height from `stp.ph0.pe` and `stp.ph1.pe` is used as the pixel intensity channel.

In [13]:
%pip install -q uproot awkward pandas plotly ipywidgets anywidget

from pathlib import Path

import awkward as ak
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import uproot

root_file = Path("/home/philip/UCL/minos/f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root")
assert root_file.exists(), f"Missing ROOT file: {root_file}"
root_file

Note: you may need to restart the kernel to use updated packages.


PosixPath('/home/philip/UCL/minos/f21048000_0000_L010185N_D07_r3.sntp.dogwood5.0.root')

In [14]:
stp_branches = {
    "stp.planeview": "NtpStRecord/stp/stp.planeview",
    "stp.strip": "NtpStRecord/stp/stp.strip",
    "stp.plane": "NtpStRecord/stp/stp.plane",
    "stp.ph0.pe": "NtpStRecord/stp/stp.ph0.pe",
    "stp.ph1.pe": "NtpStRecord/stp/stp.ph1.pe",
}

root_handle = uproot.open(root_file)
tree = root_handle["NtpSt;1"]
required_branches = list(stp_branches.values())

print("Selected tree: NtpSt;1")
print(f"Number of events: {tree.num_entries}")


Selected tree: NtpSt;1
Number of events: 119205


In [ ]:
import pandas as pd
from ipywidgets import HBox, VBox, Button, IntText
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plane_min, plane_max = 1, 485
strip_min, strip_max = 0, 191
gap_columns = 24
gap_start = 244
all_disp_planes = [p if p < gap_start else p + gap_columns for p in range(plane_min, plane_max + 1)]
gap_range = list(range(gap_start, gap_start + gap_columns))
u_raw = [p for p in range(plane_min, plane_max + 1) if (p % 2 == 0 if p < 250 else p % 2 == 1)]
v_raw = [p for p in range(plane_min, plane_max + 1) if (p % 2 == 1 if p < 250 else p % 2 == 0)]
u_planes = [p for p in u_raw if p < gap_start] + gap_range + [p + gap_columns for p in u_raw if p >= gap_start]
v_planes = [p for p in v_raw if p < gap_start] + gap_range + [p + gap_columns for p in v_raw if p >= gap_start]
full_planes = all_disp_planes
full_strips = list(range(strip_min, strip_max + 1))

# Initialize FigureWidget
fig = go.FigureWidget(make_subplots(
    rows=2, 
    cols=1, 
    vertical_spacing=0.06, 
    subplot_titles=["U View", "V View"]
))

# Add traces and formatting
for i in range(2):
    heatmap = go.Heatmap(
        z=[[0]], x=[0], y=[0],
        coloraxis="coloraxis",
        hovertemplate="plane=%{x}<br>strip=%{y}<br>PE=%{z:.2f}<extra></extra>"
    )
    fig.add_trace(heatmap, row=i+1, col=1)
    fig.add_vrect(x0=gap_start - 0.5, x1=gap_start - 0.5 + gap_columns, fillcolor="white", opacity=0.08, line_width=0, row=i+1, col=1)
    fig.add_annotation(x=(gap_start - 0.5) + gap_columns / 2, y=strip_max + 7, text="1.5 m gap", showarrow=False, font=dict(color="white", size=10), row=i+1, col=1)
    
    tick_planes = [1, 50, 100, 150, 200, 243, 244, 293, 343, 393, 443, 485]
    view_raw = [p for p in range(plane_min, plane_max + 1) if (p % 2 == 0 if p < 250 else p % 2 == 1)] if i == 0 else [p for p in range(plane_min, plane_max + 1) if (p % 2 == 1 if p < 250 else p % 2 == 0)]
    tick_positions = []
    tick_text_list = []
    for t in tick_planes:
        nearest = min(view_raw, key=lambda p: abs(p - t))
        disp_pos = nearest if nearest < gap_start else nearest + gap_columns
        tick_positions.append(disp_pos)
        tick_text_list.append(str(nearest))
    fig.update_xaxes(tickmode="array", tickvals=tick_positions, ticktext=tick_text_list, constrain="domain", row=i+1, col=1)
    fig.update_yaxes(range=[strip_min - 0.5, strip_max + 0.5], scaleanchor="x" if i == 0 else f"x{i+1}", scaleratio=2, constrain="domain", row=i+1, col=1)

fig.update_layout(
    height=900, width=850, template="plotly_dark",
    paper_bgcolor="black", plot_bgcolor="black",
    coloraxis=dict(colorscale="Turbo", cmin=0, cmax=1, colorbar=dict(title="total PE (ph0.pe + ph1.pe)")),
    margin=dict(l=60, r=10, t=40, b=60),
)

def update_plot(event_index):
    event = tree.arrays(required_branches, entry_start=event_index, entry_stop=event_index+1, library="ak")[0]
    planeview = ak.to_list(event["NtpStRecord/stp/stp.planeview"])
    strip = ak.to_list(event["NtpStRecord/stp/stp.strip"])
    plane = ak.to_list(event["NtpStRecord/stp/stp.plane"])
    ph0_pe = ak.to_list(event["NtpStRecord/stp/stp.ph0.pe"])
    ph1_pe = ak.to_list(event["NtpStRecord/stp/stp.ph1.pe"])
    total_pe = [east + west for east, west in zip(ph0_pe, ph1_pe)]
    
    vmax = max(total_pe) if total_pe else 1
    
    with fig.batch_update():
        fig.layout.coloraxis.cmax = vmax
        fig.layout.xaxis.range = full_x_range
        fig.layout.yaxis.range = full_y_range
        fig.layout.xaxis2.range = full_x_range
        fig.layout.yaxis2.range = full_y_range
        for i, view in enumerate([2, 3]):
            mask = [value == view for value in planeview]
            if not any(mask):
                view_planes = u_planes if view == 2 else v_planes
                image_values = pd.DataFrame(0, index=full_strips, columns=view_planes).values
            else:
                view_strip = [value for value, keep in zip(strip, mask) if keep]
                view_plane = [value for value, keep in zip(plane, mask) if keep]
                view_pe = [value for value, keep in zip(total_pe, mask) if keep]
                mirrored_strip = [strip_max - value for value in view_strip] if i == 1 else view_strip
                display_plane = [value if value < gap_start else value + gap_columns for value in view_plane]
                view_frame = pd.DataFrame({"plane": display_plane, "strip": mirrored_strip, "pe": view_pe})
                image = view_frame.pivot_table(index="strip", columns="plane", values="pe", aggfunc="sum", fill_value=0)
                view_planes = u_planes if view == 2 else v_planes
                image = image.reindex(index=full_strips, columns=view_planes, fill_value=0)
                image_values = image.values
                
            fig.data[i].z = image_values
            fig.data[i].x = view_planes
            fig.data[i].y = full_strips

    # Update MC Truth Info
    try:
        inu_val = ak.to_list(event["NtpStRecord/mc/mc.inu"])[0]
        iact_val = ak.to_list(event["NtpStRecord/mc/mc.iaction"])[0]
        ires_val = ak.to_list(event["NtpStRecord/mc/mc.iresonance"])[0]
        p4neu_val = ak.to_list(event["NtpStRecord/mc/mc.p4neu[4]"])[0]
        nu_str = NEUTRINO_FLAVORS.get(inu_val, f"PDG {inu_val}")
        curr_str = INTERACTION_CURRENTS.get(iact_val, f"Current {iact_val}")
        chan_str = INTERACTION_CHANNELS.get(ires_val, f"Code {ires_val}")
        e_nu_val = p4neu_val[3]
        info_label.value = f'''<div style="font-family: monospace; font-size: 13px; color: #ffffff; background-color: #1a1a1a; padding: 6px 12px; border-radius: 6px; border: 1px solid #333; margin-top: 6px; margin-bottom: 4px; display: inline-block;"><b>Neutrino:</b> <span style="color: #4fc3f7;">{nu_str}</span> &nbsp;|&nbsp; <b>Type:</b> <span style="color: #81c784;">{curr_str}</span> &nbsp;|&nbsp; <b>Channel:</b> <span style="color: #ffb74d;">{chan_str}</span> &nbsp;|&nbsp; <b>True E<sub>ν</sub>:</b> <span style="color: #ba68c8;">{e_nu_val:.2f} GeV</span></div>'''
    except Exception as e:
        info_label.value = "<div style='color: #888;'>MC Truth metadata unavailable</div>"

info_label = HTML(value="")

NEUTRINO_FLAVORS = {14: "ν_μ", -14: "ν̄_μ", 12: "ν_e", -12: "ν̄_e", 16: "ν_τ", -16: "ν̄_τ"}
INTERACTION_CURRENTS = {1: "CC (Charged Current)", 0: "NC (Neutral Current)"}
INTERACTION_CHANNELS = {1001: "QE (Quasi-Elastic)", 1002: "RES (Resonant / Single Pion)", 1003: "DIS (Deep Inelastic Scattering)", 1004: "COH (Coherent)"}

event_input = IntText(value=0, description='Event:', layout={'width': '200px'})
btn_prev = Button(description='Prev', icon='backward')
btn_next = Button(description='Next', icon='forward')

def on_prev(b):
    event_input.value = max(0, event_input.value - 1)
def on_next(b):
    event_input.value = min(tree.num_entries - 1, event_input.value + 1)
def on_value_change(change):
    update_plot(change['new'])

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
event_input.observe(on_value_change, names='value')

is_syncing = False
y_center = (strip_min + strip_max) / 2.0

import numpy as np
is_syncing = False

def get_y_mid(trace_index, x0, x1):
    try:
        z_data = np.array(fig.data[trace_index].z)
        view_planes = u_planes if trace_index == 0 else v_planes
        cols_mask = (np.array(view_planes) >= x0) & (np.array(view_planes) <= x1)
        sub_z = z_data[:, cols_mask]
        hit_rows = np.where(sub_z > 0)[0]
        if len(hit_rows) > 0:
            return float((hit_rows.min() + hit_rows.max()) / 2.0)
    except Exception:
        pass
    return (strip_min + strip_max) / 2.0

full_x_range = [full_planes[0] - 0.5, full_planes[-1] + 0.5]
full_y_range = [strip_min - 0.5, strip_max + 0.5]

def sync_from_plot1(layout, xrange, yrange, xauto=None):
    global is_syncing
    if is_syncing: return
    is_syncing = True
    try:
        with fig.batch_update():
            is_full_view = xauto is True or not xrange or len(xrange) != 2 or (xrange[1] - xrange[0] >= 400)
            if is_full_view:
                fig.layout.xaxis.range = full_x_range
                fig.layout.yaxis.range = full_y_range
                fig.layout.xaxis2.range = full_x_range
                fig.layout.yaxis2.range = full_y_range
            else:
                x0, x1 = xrange[0], xrange[1]
                fig.layout.xaxis2.range = [x0, x1]
                y_span = (yrange[1] - yrange[0]) if (yrange and len(yrange) == 2) else (x1 - x0)
                y_mid = get_y_mid(1, x0, x1)
                fig.layout.yaxis2.range = [y_mid - y_span/2.0, y_mid + y_span/2.0]
    finally:
        is_syncing = False

def sync_from_plot2(layout, xrange, yrange, xauto=None):
    global is_syncing
    if is_syncing: return
    is_syncing = True
    try:
        with fig.batch_update():
            is_full_view = xauto is True or not xrange or len(xrange) != 2 or (xrange[1] - xrange[0] >= 400)
            if is_full_view:
                fig.layout.xaxis.range = full_x_range
                fig.layout.yaxis.range = full_y_range
                fig.layout.xaxis2.range = full_x_range
                fig.layout.yaxis2.range = full_y_range
            else:
                x0, x1 = xrange[0], xrange[1]
                fig.layout.xaxis.range = [x0, x1]
                y_span = (yrange[1] - yrange[0]) if (yrange and len(yrange) == 2) else (x1 - x0)
                y_mid = get_y_mid(0, x0, x1)
                fig.layout.yaxis.range = [y_mid - y_span/2.0, y_mid + y_span/2.0]
    finally:
        is_syncing = False

fig.layout.on_change(sync_from_plot1, "xaxis.range", "yaxis.range", "xaxis.autorange")
fig.layout.on_change(sync_from_plot2, "xaxis2.range", "yaxis2.range", "xaxis2.autorange")

# Load the initial event
update_plot(0)

# Display UI without white background artifacts
VBox([HBox([btn_prev, event_input, btn_next]), info_label, fig])

